In [ ]:
import torch
import numpy as np

from utils import *

In [ ]:
if torch.cuda.is_available():
    device = 'cuda:0'
else:
    device = 'cpu'

# dataset
dataset = "syn"
print(dataset)

d1 = 10000
d2 = 1000
r = 10
M = load_data_syn(r, d1, d2, device)

# privacy
delta = 1/d1
epsilon = 10
sigma = torch.sqrt(2*torch.log(torch.tensor(1.25/delta)))/epsilon

# main part
err_list = []
rmse_list = []    
p = 0.01

# sample 2 entries per row
#observed_M, masks = get_random_samples_per_row(M.cpu().numpy(), 2)
#observed_M = torch.from_numpy(observed_M).float().to(device)
#masks = torch.from_numpy(masks).to(device)

# IID sample
observed_M, masks = get_uniform_masks(M, p)

# observed second-moment matrix
MTM = M.T @ M
cov_observe_M =  observed_M.T @ observed_M


In [ ]:
# Inverse estimated probability weighting & privacy injection
noise_matrix = sym_noise(d2, sigma).to(device)
cov_observe_count = (1 * (observed_M != 0)).float().T @ (1 * (observed_M != 0).float())
cov_observe_count = cov_observe_count + (cov_observe_count == 0) * 1

p_hat = (observed_M!=0).sum()/(d1*d2)

T_masks = 1 * (cov_observe_M!=0)
#cov_observe_M += noise_matrix
ratio = T_masks.sum() / d2**2

T = cov_observe_M / (cov_observe_count/d1)

# calculate matrix of p_hat and p
p_hat_matrix = (cov_observe_count/d1)
p_matrix = (torch.diag(torch.diag(torch.ones(d2, d2)*(p-p**2))) + torch.ones(d2, d2)*p**2).to(device)
#print(p_hat_matrix)
#print(p_matrix)

# calculate (p_hat - p)
print((p_matrix*T_masks-p_hat_matrix*T_masks).mean()/ratio)
# calculate the relative err of T_p_hat
print(torch.norm(T*T_masks-MTM*T_masks)/torch.norm(MTM*T_masks))

In [ ]:
# calculate Taylor expansion
diag_cov = torch.diag( torch.diag(cov_observe_M) )
T_p = ((1.0 / p) * diag_cov + (1.0 / (p**2)) * (cov_observe_M - diag_cov))
T_p_1 = ((-1.0 / p**2) * diag_cov - (2.0 / (p**3)) * (cov_observe_M - diag_cov))
T_p_2 = ((2.0 / p**3) * diag_cov + (6.0 / (p**4)) * (cov_observe_M - diag_cov))/2
T_p_3 = ((-6.0 / p**4) * diag_cov - (24.0 / (p**5)) * (cov_observe_M - diag_cov))

item_1 = torch.mul(T_p_1,(p_hat_matrix-p_matrix))
item_2 = torch.mul(T_p_2,(p_hat_matrix-p_matrix)**2)
item_3 = torch.mul(T_p_3,(p_hat_matrix-p_matrix)**3)

#mask_err_Tp = T_p[T_masks] - MTM[T_masks]
#print(MTM)
#print(T_p)
#print(T_p_1)
#print(T_p_2)

# the result of each item in the Taylor expansion
print((item_1*T_masks).mean()/ratio)
print((item_2*T_masks).mean()/ratio)
print((item_3*T_masks).mean()/ratio)
# calculate the relative err of T_p_hat
print(torch.norm(T_p*T_masks - MTM*T_masks)/torch.norm(MTM*T_masks))